# EDA reproducible de Aeroméxico

## tl;dr

La historia mensual de pasajeros es apta para modelado sencillo; la historia trimestral de Aeroméxico todavía debe tratarse como descriptiva. COVID permanece en la serie y no se reemplazan faltantes por cero.

## Context & Methods

Este notebook es un compañero auditable. La lógica productiva vive en `src.analytics.eda`; aquí se ejecuta y se muestran resultados acotados.

### Key Assumptions

- Vista consolidada de Aeroméxico.
- Periodicidad mensual para pasajeros.
- STL de periodo 12.
- COVID se conserva como régimen extraordinario.

In [1]:
from src.analytics.eda import run_eda
eda = run_eda()
{k: (len(v) if hasattr(v, '__len__') and not isinstance(v, dict) else v) for k, v in eda.items()}

{'coverage': 247,
 'descriptives': 247,
 'seasonality': 138,
 'correlations': 28,
 'structural_breaks': 12,
 'seasonal_stats': {'observations': 138,
  'seasonal_amplitude_passengers': 627661.5383355084,
  'seasonal_amplitude_pct_of_median_trend': 0.44954721665293396,
  'strongest_month': 7,
  'weakest_month': 2},
 'covid_policy': 26}

## Data

### 1. Cobertura de la serie objetivo

In [2]:
coverage = eda['coverage']
coverage.query("carrier_key == 'AEROMEXICO' and metric_key == 'passengers_afac'")[['segment','observations','first_period','last_period','null_values']]

,segment,observations,first_period,last_period,null_values
59,domestic,138,2015M01,2026M06,0.0
60,international,138,2015M01,2026M06,0.0
61,total,138,2015M01,2026M06,0.0


## Results

### 2. Tendencia y estacionalidad mensual

In [3]:
import plotly.graph_objects as go
seasonality = eda['seasonality']
fig = go.Figure()
fig.add_scatter(x=seasonality['period_id'], y=seasonality['observed'], name='Observado', line=dict(color='#0B3A66'))
fig.add_scatter(x=seasonality['period_id'], y=seasonality['trend'], name='Tendencia', line=dict(color='#C89211', dash='dash'))
fig.update_layout(title='Pasajeros AFAC observados y tendencia STL', xaxis_title='Mes', yaxis_title='Pasajeros', template='plotly_white')
fig.show()

### 3. Relaciones con variables macro, con rezagos

In [4]:
corr = eda['correlations'].dropna().assign(abs_corr=lambda d: d.correlation.abs())
corr.sort_values('abs_corr', ascending=False).head(12)[['indicator_key','lag_months','correlation','observations']]

,indicator_key,lag_months,correlation,observations
6,inpc,6,0.751336,132
5,inpc,5,0.744568,133
4,inpc,4,0.735376,134
3,inpc,3,0.727999,135
2,inpc,2,0.720098,136
1,inpc,1,0.718237,137
0,inpc,0,0.713571,138
9,jet_fuel_usd_per_gallon,2,0.699216,136
8,jet_fuel_usd_per_gallon,1,0.697901,137
7,jet_fuel_usd_per_gallon,0,0.681540,138


### 4. Quiebres estructurales candidatos

In [5]:
eda['structural_breaks'].head(10)

,break_period,standardized_mean_shift,rank,is_known_regime
62,2022M03,1.144168,1,True
24,2019M01,1.097527,2,False
63,2022M04,1.087353,3,True
27,2019M04,1.079944,4,False
25,2019M02,1.079507,5,False
26,2019M03,1.078791,6,False
38,2020M03,1.070272,7,True
28,2019M05,1.066725,8,False
29,2019M06,1.063194,9,False
61,2022M02,1.040653,10,True


## Takeaways

- Los pasajeros mensuales son la única prioridad de forecast con historia larga y completa.
- La estacionalidad debe estar dentro del baseline; una comparación contra el mes anterior sería insuficiente.
- Los quiebres alrededor de 2020-2023 se conservan como parte de la historia.
- Correlación no implica causalidad y los rezagos macro son descriptivos.
- Las conclusiones de negocio completas están en `docs/analytics/eda-hallazgos.md`.